# A2 Evaluation and Robustness Summary

This notebook is for Member A's later-stage integration work.

It should be run after B/C/D provide their prediction CSV files.

Purpose:

1. Load clean prediction CSV files from B/C/D.
2. Run unified evaluation.
3. Compare model performance.
4. Summarise robustness metrics.
5. Plot robustness curves.

This notebook depends on the shared scripts in `src/`.


## 1. Setup

In [ ]:
from pathlib import Path
import sys
import json
import subprocess

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

## 2. Expected clean prediction files

B/C/D should provide these files:

- `results/clean/traditional_predictions.csv`
- `results/clean/scratch_predictions.csv`
- `results/clean/pretrained_predictions.csv`

Each file should have this format:

```text
image_path,true_idx,pred_idx,top5_idx
```


In [ ]:
clean_dir = PROJECT_ROOT / "results" / "clean"

prediction_files = {
    "traditional": clean_dir / "traditional_predictions.csv",
    "scratch": clean_dir / "scratch_predictions.csv",
    "pretrained": clean_dir / "pretrained_predictions.csv",
}

for model, path in prediction_files.items():
    print(model, "exists:", path.exists(), "|", path)

## 3. Validate prediction CSV format

This step checks whether the prediction files exist and contain the required columns.


In [ ]:
required_cols = {"image_path", "true_idx", "pred_idx", "top5_idx"}

for model, path in prediction_files.items():
    if not path.exists():
        print(f"[MISSING] {model}: {path}")
        continue

    df = pd.read_csv(path)
    missing = required_cols - set(df.columns)
    print(f"===== {model} =====")
    print("rows:", len(df))
    print("columns:", list(df.columns))
    print("missing required columns:", missing)
    display(df.head())

## 4. Run unified clean evaluation

This uses `src/evaluate.py` to generate:

- `results/clean/traditional_metrics.json`
- `results/clean/scratch_metrics.json`
- `results/clean/pretrained_metrics.json`


In [ ]:
def run_evaluate(pred_csv, out_json, num_classes=500):
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "src" / "evaluate.py"),
        "--pred-csv", str(pred_csv),
        "--out-json", str(out_json),
        "--num-classes", str(num_classes),
    ]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result.returncode

for model, pred_path in prediction_files.items():
    if pred_path.exists():
        out_path = clean_dir / f"{model}_metrics.json"
        run_evaluate(pred_path, out_path)
    else:
        print(f"Skip {model}: prediction file not found")

## 5. Load clean metrics and create comparison table

In [ ]:
metric_files = {
    "traditional": clean_dir / "traditional_metrics.json",
    "scratch": clean_dir / "scratch_metrics.json",
    "pretrained": clean_dir / "pretrained_metrics.json",
}

rows = []
for model, path in metric_files.items():
    if not path.exists():
        print(f"[MISSING] {model}: {path}")
        continue

    with open(path, "r", encoding="utf-8") as f:
        metrics = json.load(f)

    rows.append({
        "model": model,
        "top1_accuracy": metrics.get("top1_accuracy"),
        "top5_accuracy": metrics.get("top5_accuracy"),
        "macro_precision": metrics.get("macro_precision"),
        "macro_recall": metrics.get("macro_recall"),
        "macro_f1": metrics.get("macro_f1"),
    })

clean_summary = pd.DataFrame(rows)
display(clean_summary)

out_csv = clean_dir / "clean_model_comparison.csv"
if len(clean_summary) > 0:
    clean_summary.to_csv(out_csv, index=False)
    print("Saved:", out_csv)

## 6. Plot clean model comparison

This creates simple bar charts for top-1 accuracy and macro-F1.


In [ ]:
fig_dir = PROJECT_ROOT / "results" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

if len(clean_summary) > 0:
    ax = clean_summary.plot(x="model", y="top1_accuracy", kind="bar", legend=False, figsize=(7, 4))
    ax.set_ylabel("Top-1 accuracy")
    ax.set_title("Clean Test Top-1 Accuracy")
    plt.tight_layout()
    plt.savefig(fig_dir / "clean_top1_accuracy.png", dpi=200)
    plt.show()

    ax = clean_summary.plot(x="model", y="macro_f1", kind="bar", legend=False, figsize=(7, 4))
    ax.set_ylabel("Macro F1")
    ax.set_title("Clean Test Macro F1")
    plt.tight_layout()
    plt.savefig(fig_dir / "clean_macro_f1.png", dpi=200)
    plt.show()
else:
    print("No clean metrics available yet.")

## 7. Robustness summary

After degraded metrics JSON files are available under `results/robustness/metrics/`, run `src/summarize_robustness.py` to create:

`results/robustness/combined_robustness_results.csv`


In [ ]:
robustness_dir = PROJECT_ROOT / "results" / "robustness"
metrics_dir = robustness_dir / "metrics"
combined_csv = robustness_dir / "combined_robustness_results.csv"

cmd = [
    sys.executable,
    str(PROJECT_ROOT / "src" / "summarize_robustness.py"),
    "--clean-dir", str(clean_dir),
    "--metrics-dir", str(metrics_dir),
    "--out-csv", str(combined_csv),
]

if metrics_dir.exists():
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
else:
    print("Robustness metrics directory does not exist yet:", metrics_dir)
    print("This is normal before B/C/D provide degraded prediction results.")

## 8. Load combined robustness results

In [ ]:
if combined_csv.exists():
    robustness_summary = pd.read_csv(combined_csv)
    display(robustness_summary.head(20))
    print("Rows:", len(robustness_summary))
else:
    robustness_summary = pd.DataFrame()
    print("Combined robustness CSV not found yet:", combined_csv)

## 9. Plot robustness curves

This uses `src/plot_robustness_curves.py`.


In [ ]:
if combined_csv.exists():
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "src" / "plot_robustness_curves.py"),
        "--input-csv", str(combined_csv),
        "--out-dir", str(fig_dir),
    ]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
else:
    print("Skip plotting: combined robustness CSV not found yet.")

## 10. Summary

This notebook will become complete after B/C/D provide their model prediction files.

Expected final outputs from this notebook:

- `results/clean/*_metrics.json`
- `results/clean/clean_model_comparison.csv`
- `results/robustness/combined_robustness_results.csv`
- clean comparison figures
- robustness figures

For the final code submission, remember that result images, trained model weights, and raw datasets should not be included in the zip file.
